In [11]:
# =============================================================================
# CELL 1: CONFIGURATION
# =============================================================================
# STE RAGU V3 Production Output -- Individual GL/Recovery first, then aggregate.
# Dual-path: MS/LTV/APR from SFS contract chain; GL/Recovery from ULA per account.
#   RAGU = MS + GL + Recovery + LTV + APR  (higher score = lower loss)
#   - GL: (1-LM) * UL/0.027, where UL = 0.9 - (MS-125)*0.027 - (72-term)/240
#   - Recovery: UL * F * R * 100  (ADDED -- higher recovery = lower net loss = higher RAGU)
#   - LTV: LTV_COEF * (baseline - actual)   [from SFS dual-path]
#   - APR: (baseline - actual)/0.01 * APR_MULT  [from SFS dual-path]

# --- Granularity: monthly only for STE ---
granularity = 'm'

# --- Date Range (STE starts October 2025) ---
START_DATE = '2025-10-07'
END_DATE = None

# --- Query Control ---
run_every_query = True

# --- Date Column ---
DATE_COL_MAP = {'m': 'book_date'}

# --- LOBs ---
LOBS = ['STE']

# --- Rollup Groups (none -- single LOB) ---
ROLLUP_GROUPS = {}

# --- Baselines (STE-specific) ---
BASELINES = {
    'STE': {'ltv': 1.94, 'apr': 0.25},
}

# --- V3 Formula Parameters ---
UL_TO_MS = 0.027
UL_INTERCEPT = 0.9
UL_MS_CENTER = 125
UL_TERM_CENTER = 72
UL_TERM_DENOM = 240
RECOVERY_TO_SCORE = 100

APR_MULT = 0.8223
FIND_RATE = 0.75
LTV_COEF = 1.1

# --- STE: No DLA (uniform pricing scalar) ---
PRICING_SCALAR = 1.0

# --- Dual-path override: MANDATORY for STE ---
USE_STE_METRICS = True

# --- Excel Output ---
EXCEL_OUTPUT = '../output/ste_ragu_v3_production.xlsx'

In [12]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import os
import openpyxl
from tqdm.notebook import tqdm

tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

PERIOD_FREQ_MAP = {'m': 'M'}
period_freq = PERIOD_FREQ_MAP[granularity]

start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)

min_date_sql = f"'{START_DATE}'"

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")
print(f"SQL min_date: {min_date_sql}")

Granularity: m
Date column: book_date
Period range: 2025-10 to 2026-08
SQL min_date: '2025-10-07'


In [13]:
# =============================================================================
# CELL 3: RAGU V3 FORMULA DEFINITIONS
# =============================================================================

def compute_unit_loss(model_score, con_term):
    """UL = 0.9 - (MS - 125)*0.027 - (72 - term)/240"""
    return UL_INTERCEPT - (model_score - UL_MS_CENTER) * UL_TO_MS - (UL_TERM_CENTER - con_term) / UL_TERM_DENOM


def compute_gross_loss_impact(loss_multiplier, unit_loss):
    """GL impact: (1-LM) * UL / 0.027"""
    return (1 - loss_multiplier) * unit_loss / UL_TO_MS


def compute_recovery_impact(recovery_multiplier, unit_loss, find_rate=FIND_RATE):
    """Recovery impact: UL * F * R * 100"""
    return unit_loss * find_rate * recovery_multiplier * RECOVERY_TO_SCORE


def compute_ltv_impact(baseline_ltv, actual_ltv, ltv_coef=LTV_COEF):
    """LTV impact: LTV_COEF * (baseline - actual)"""
    return ltv_coef * (baseline_ltv - actual_ltv)


def compute_apr_impact(baseline_apr, actual_apr, apr_mult=APR_MULT):
    """APR impact: per 1% deviation from baseline."""
    return (baseline_apr - actual_apr) / 0.01 * apr_mult

In [14]:
# =============================================================================
# CELL 4: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False):
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection)
        if df.empty:
            print(f"  WARNING: query '{filename}' returned 0 rows")
        store_pickle(df, pickle_name)
        return df
    df = get_pickle(pickle_name)
    if df.empty:
        print(f"  WARNING: cached '{pickle_name}' contains 0 rows -- consider refreshing")
    return df


def weighted_average_and_sum(group, metrics):
    """Per-metric population-aware weighted average."""
    if isinstance(metrics, str):
        valid = group[metrics].notna()
        if valid.any():
            weighted_avg = (group.loc[valid, metrics] * group.loc[valid, 'amt_financed']).sum() / group.loc[valid, 'amt_financed'].sum()
        else:
            weighted_avg = np.nan
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        valid = group[metric].notna()
        if valid.any():
            result_dict[metric] = (
                (group.loc[valid, metric] * group.loc[valid, 'amt_financed']).sum()
                / group.loc[valid, 'amt_financed'].sum()
            )
        else:
            result_dict[metric] = np.nan
    return pd.Series(result_dict)


def format_vintage(period_series):
    """Convert pd.Period series to formatted vintage strings."""
    if len(period_series) == 0:
        return period_series.astype(str)
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)

In [15]:
# =============================================================================
# CELL 5: ULA MULTIPLIER FUNCTIONS (NonKMX only -- STE is nonKMX)
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag

    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag

    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)

    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)

    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag

    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag

    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date

    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag

    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag

    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)

    return ula_df

In [16]:
# =============================================================================
# CELL 6: DATA FETCH (SQL + PICKLE)
# =============================================================================

os.makedirs('cache', exist_ok=True)
force = run_every_query

need_conn = force or not all(
    os.path.exists(p) for p in ('../../cache/ste_ula_v1.pkl', '../../cache/ste_recovery_v1.pkl', '../../cache/ste_weekly_v1.pkl')
)

if need_conn:
    conn = pyodbc.connect("DSN=Redshift_prod_new")

    with open('../queries/ste_ragu_temptables.txt', 'r') as f:
        conn.execute(f.read().strip())
    print('Temp tables created')

    ula_df_total = cached_sql('../queries/ste_ragu_ula.txt', '../../cache/ste_ula_v1.pkl',
                              connection=conn, force_refresh=force)
    print(f'ULA ready: {len(ula_df_total):,} records')

    new_recovery = cached_sql('../queries/ste_ragu_recovery.txt', '../../cache/ste_recovery_v1.pkl',
                              connection=conn, force_refresh=force)
    print(f'Recovery ready: {len(new_recovery):,} records')

    ste_weekly_raw = cached_sql('../queries/ste_ragu_weekly.txt', '../../cache/ste_weekly_v1.pkl',
                                connection=conn, force_refresh=force)
    print(f'STE weekly metrics ready: {len(ste_weekly_raw):,} records')

    conn.close()
else:
    ula_df_total = get_pickle('../../cache/ste_ula_v1.pkl')
    new_recovery = get_pickle('../../cache/ste_recovery_v1.pkl')
    ste_weekly_raw = get_pickle('../../cache/ste_weekly_v1.pkl')
    print('All data loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")

Temp tables created
ULA ready: 138,192 records
Recovery ready: 21,354 records
STE weekly metrics ready: 21,436 records
ULA records: 138,192


In [17]:
# =============================================================================
# CELL 7: DATA PREP, FLAG CREATION, AND STE METRICS (DUAL-PATH)
# =============================================================================

# --- Filter and relabel ---
ula_df_total = ula_df_total[ula_df_total.lob != 'Core']
ula_df_total['lob'] = 'STE'

# --- Date parsing ---
ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

# --- Period assignment ---
for df in [ula_df_total, new_recovery]:
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['period'] = df['month']

# --- Filter to configured date range ---
for df in [ula_df_total, new_recovery]:
    mask = (df['period'] >= start_period) & (df['period'] <= end_period)
    df.drop(df[~mask].index, inplace=True)

# --- String version of date_col ---
ula_df_total['book_week'] = ula_df_total['book_week'].astype(str)
date_col_str = f'{date_col}_str'
ula_df_total[date_col_str] = ula_df_total[date_col].astype(str)

# --- STE-specific caps ---
ula_df_total['bbltv'] = ula_df_total.amt_financed / ula_df_total.bbvalue.replace(0, np.nan)
ula_df_total = ula_df_total[
    (ula_df_total.bbltv <= 10.0) |
    (ula_df_total.bbvalue.isna()) |
    (ula_df_total.bbvalue == 0)
]
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]

print(f"Periods in data: {ula_df_total['period'].nunique()}")
print(f"Period range: {ula_df_total['period'].min()} to {ula_df_total['period'].max()}")
print(f"ULA after caps: {len(ula_df_total):,}")

# --- ULA Processing ---
ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# --- No DLA: uniform pricing scalar ---
ula_df_total['pricing_scalar'] = PRICING_SCALAR

# --- Driver Flag ---
warnings.filterwarnings("ignore", category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings("default", category=UserWarning)

# --- NA Handling ---
ula_df_total = ula_df_total.dropna(subset=['lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# --- NonKMX Flags (STE uses nonKMX path with PTI threshold = 0.2) ---
ula_df_total['ent_fld_flag'] = False
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue > 0) & (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500)
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130)
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = False
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1 if 'seasonal_employment_flag' in ula_df_total.columns else (ula_df_total.get('employment', '') == 'seasonal')
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1 if 'waiter_employment_flag' in ula_df_total.columns else (ula_df_total.get('employment', '') == 'waiter')
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = False
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = 0

# --- Deduplicate driver flags ---
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# --- Vintage strings ---
ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

print(f"ULA after refinement: {len(ula_df_total):,}")

# --- DUAL-PATH: Build ste_metrics_df from ste_ragu_weekly.txt ---
ste_filtered = ste_weekly_raw.copy()
ste_date_col = 'book_date'
ste_filtered[ste_date_col] = pd.to_datetime(ste_filtered[ste_date_col])
ste_filtered['period'] = ste_filtered[ste_date_col].dt.to_period(period_freq)
ste_filtered = ste_filtered[(ste_filtered.period >= start_period) & (ste_filtered.period <= end_period)]
ste_filtered['vintage'] = format_vintage(ste_filtered['period'])

# Apply same STE caps
ste_filtered = ste_filtered[ste_filtered['con_pti_back'] <= 0.6]
ste_filtered = ste_filtered[ste_filtered['total_income'] <= 200000]
ste_filtered['bbltv'] = ste_filtered['con_amount_financed_back'] / ste_filtered['bb_value'].replace(0, np.nan)
ste_filtered = ste_filtered[
    (ste_filtered['bbltv'] <= 10.0) |
    (ste_filtered['bb_value'].isna()) |
    (ste_filtered['bb_value'] == 0)
]

def _build_ste_metrics(g):
    w = g['con_amount_financed_back']
    ms_valid = g['con_risk_model_score'].notnull()
    ltv_valid = g['bbltv'].notnull()
    return pd.Series({
        'model_score_wtd': (g.loc[ms_valid, 'con_risk_model_score'] * w[ms_valid]).sum() / w[ms_valid].sum() if ms_valid.any() else np.nan,
        'ltv_wtd': (g.loc[ltv_valid, 'bbltv'] * w[ltv_valid]).sum() / w[ltv_valid].sum() if ltv_valid.any() else np.nan,
        'apr_wtd': (g['con_apr'] * w).sum() / w.sum(),
        'amt_financed_total': w.sum(),
        'n_accounts': len(g),
    })

ste_metrics_df = ste_filtered.groupby('vintage').apply(_build_ste_metrics).reset_index()
print(f"\nste_metrics_df built: {len(ste_metrics_df)} vintages")
print(ste_metrics_df.head(10))

Periods in data: 11
Period range: 2025-10 to 2026-08
ULA after caps: 134,947
ULA after refinement: 20,099

ste_metrics_df built: 11 vintages
    vintage  model_score_wtd   ltv_wtd   apr_wtd  amt_financed_total  \
0  2025 M10       131.021335  1.522845  0.236499        3.270728e+07   
1  2025 M11       131.365156  1.544695  0.234064        4.813154e+07   
2  2025 M12       131.651392  1.573478  0.234407        4.906867e+07   
3  2026 M01       132.250448  1.538778  0.234458        4.206601e+07   
4  2026 M02       133.111325  1.500602  0.233213        5.189566e+07   
5  2026 M03       134.393443  1.461419  0.228505        1.078094e+08   
6  2026 M04       135.073919  1.446427  0.226109        9.053636e+07   
7  2026 M05       136.244091  1.408017  0.224527        7.272996e+07   
8  2026 M06       136.526976  1.423415  0.223104        8.233838e+07   
9  2026 M07       137.337138  1.411365  0.220733        5.940834e+07   

   n_accounts  
0       974.0  
1      1531.0  
2      1600.0  
3 

In [18]:
# =============================================================================
# CELL 8: ACCOUNT-LEVEL V3 SCORING (GL + Recovery per account)
# =============================================================================

# --- Apply ULA multipliers (nonKMX only for STE) ---
acct_df = get_ula_multiplier_nonkmx(ula_df_total.copy())

# --- Merge recovery multiplier ---
nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(
    subset='account_number', keep='first')
acct_df = acct_df.merge(
    nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
    on='account_number', how='left'
).drop_duplicates(subset='account_number', keep='first')

has_bb = acct_df['bbvalue'].notna() & (acct_df['bbvalue'] > 0)
print(f"Total accounts: {len(acct_df):,}")
print(f"  with bbvalue > 0: {has_bb.sum():,}")
print(f"  with recovery: {acct_df['recovery_multiplier'].notna().sum():,}")

# --- Per-account V3 formula (con_term unavailable; defaults to 72) ---
acct_df['model_score'] = acct_df.cd_model_score
acct_df['ltv'] = np.where(has_bb, acct_df.amt_financed / acct_df.bbvalue, np.nan)
acct_df['con_term_filled'] = UL_TERM_CENTER  # STE ULA query does not provide con_term

acct_df['unit_loss'] = compute_unit_loss(acct_df['model_score'], acct_df['con_term_filled'])
acct_df['gross_loss_impact'] = compute_gross_loss_impact(acct_df['loss_multiplier'], acct_df['unit_loss'])
acct_df['recovery_impact'] = compute_recovery_impact(acct_df['recovery_multiplier'], acct_df['unit_loss'])

print(f"\nV3 per-account scoring complete.")
print(f"  GL impact range: [{acct_df['gross_loss_impact'].min():.2f}, {acct_df['gross_loss_impact'].max():.2f}]")
print(f"  Recovery impact: {acct_df['recovery_impact'].notna().sum():,} accounts scored")
print(f"  Recovery impact range: [{acct_df['recovery_impact'].min():.2f}, {acct_df['recovery_impact'].max():.2f}]")

acct_df.head(10)[['account_number', 'vintage', 'model_score', 'loss_multiplier',
                   'unit_loss', 'gross_loss_impact', 'recovery_impact', 'amt_financed']]

Total accounts: 20,099
  with bbvalue > 0: 19,980
  with recovery: 19,095

V3 per-account scoring complete.
  GL impact range: [-15.35, 5.45]
  Recovery impact: 19,095 accounts scored
  Recovery impact range: [-20.84, 62.70]


,account_number,vintage,model_score,loss_multiplier,unit_loss,gross_loss_impact,recovery_impact,amt_financed
0,9.012514e+10,2025 M12,133.0,1.155863,0.684,-3.948532,33.340352,73519.69
1,9.012516e+10,2026 M01,134.0,0.961672,0.657,0.932640,29.594867,28420.00
2,9.012520e+10,2026 M03,132.0,0.973496,0.711,0.697934,22.860453,25927.91
3,9.012513e+10,2025 M12,140.0,0.890169,0.495,2.013568,18.062253,34914.08
4,9.012524e+10,2026 M04,135.0,0.979186,0.630,0.485662,26.539573,23838.29
5,9.012511e+10,2025 M11,128.0,0.985320,0.819,0.445293,32.351886,41818.87
6,9.012510e+10,2025 M10,125.0,1.263780,0.900,-8.792667,45.715196,29082.50
7,9.012514e+10,2025 M12,127.0,0.917700,0.846,2.578733,31.575020,25547.86
8,9.012515e+10,2026 M01,145.0,0.955760,0.360,0.589861,17.954557,64267.94
9,9.012532e+10,2026 M07,138.0,1.141738,0.549,-2.882016,27.127162,22542.59


In [19]:
# =============================================================================
# CELL 9: AGGREGATION + DUAL-PATH OVERRIDE
# =============================================================================
# Aggregate GL/Recovery from per-account scoring, then override MS/LTV/APR
# from SFS dual-path (ste_metrics_df).

def population_aware_agg(grp):
    """Aggregate per-account metrics to vintage level."""
    result = {}

    full_pop = grp
    bb_pop = grp[grp['ltv'].notna()]
    scored_pop = grp[grp['recovery_impact'].notna() & grp['ltv'].notna()]

    for m in ['model_score', 'gross_loss_impact']:
        sub = full_pop[full_pop[m].notna()]
        if len(sub) > 0:
            result[m] = (sub[m] * sub['amt_financed']).sum() / sub['amt_financed'].sum()
        else:
            result[m] = np.nan

    for m in ['recovery_impact']:
        sub = scored_pop[scored_pop[m].notna()]
        if len(sub) > 0:
            result[m] = (sub[m] * sub['amt_financed']).sum() / sub['amt_financed'].sum()
        else:
            result[m] = np.nan

    result['amt_financed'] = full_pop['amt_financed'].sum()
    return pd.Series(result)


vintage_df = acct_df.groupby('vintage').apply(
    population_aware_agg, include_groups=False
).reset_index()

print(f"Account-level aggregation: {len(vintage_df)} vintages")

# --- Dual-path override: MS, LTV, APR from SFS ---
vintage_df = vintage_df.merge(ste_metrics_df[['vintage', 'model_score_wtd', 'ltv_wtd', 'apr_wtd']],
                              on='vintage', how='left')

vintage_df['model_score'] = vintage_df['model_score_wtd']
vintage_df['ltv'] = vintage_df['ltv_wtd']
vintage_df['apr'] = vintage_df['apr_wtd']

# --- Compute LTV and APR impacts from SFS dual-path values ---
baseline_ltv = BASELINES['STE']['ltv']
baseline_apr = BASELINES['STE']['apr']

vintage_df['ltv_impact'] = compute_ltv_impact(baseline_ltv, vintage_df['ltv'])
vintage_df['apr_impact'] = compute_apr_impact(baseline_apr, vintage_df['apr'])

# --- Final RAGU = MS + GL + Recovery + LTV + APR ---
vintage_df['ragu_score'] = (
    vintage_df['model_score']
    + vintage_df['gross_loss_impact']
    + vintage_df['recovery_impact']
    + vintage_df['ltv_impact']
    + vintage_df['apr_impact']
)

vintage_df['lob'] = 'STE'

# --- Cleanup helper columns ---
vintage_df = vintage_df.drop(columns=['model_score_wtd', 'ltv_wtd', 'apr_wtd'], errors='ignore')

# --- Verification ---
print(f"\nVerification (RAGU = MS + GL + Recovery + LTV + APR):")
for _, row in vintage_df.head(5).iterrows():
    check = row['model_score'] + row['gross_loss_impact'] + row['recovery_impact'] + row['ltv_impact'] + row['apr_impact']
    print(f"  {row['vintage']}: {row['model_score']:.2f} + {row['gross_loss_impact']:.2f} "
          f"+ {row['recovery_impact']:.2f} + {row['ltv_impact']:.2f} + {row['apr_impact']:.2f} "
          f"= {check:.2f} (stored: {row['ragu_score']:.2f})")

print(f"\nFinal output: {len(vintage_df)} rows")
vintage_df[['vintage', 'lob', 'model_score', 'gross_loss_impact', 'recovery_impact',
            'ltv_impact', 'apr_impact', 'ragu_score', 'ltv', 'apr', 'amt_financed']]

Account-level aggregation: 11 vintages

Verification (RAGU = MS + GL + Recovery + LTV + APR):
  2025 M10: 131.02 + -0.52 + 33.37 + 0.46 + 1.11 = 165.44 (stored: 165.44)
  2025 M11: 131.37 + -0.45 + 32.20 + 0.43 + 1.31 = 164.85 (stored: 164.85)
  2025 M12: 131.65 + -0.56 + 31.74 + 0.40 + 1.28 = 164.51 (stored: 164.51)
  2026 M01: 132.25 + -0.48 + 31.02 + 0.44 + 1.28 = 164.51 (stored: 164.51)
  2026 M02: 133.11 + -0.50 + 29.28 + 0.48 + 1.38 = 163.76 (stored: 163.76)

Final output: 11 rows


,vintage,lob,model_score,gross_loss_impact,recovery_impact,ltv_impact,apr_impact,ragu_score,ltv,apr,amt_financed
0,2025 M10,STE,131.021335,-0.520985,33.366306,0.458871,1.110161,165.435687,1.522845,0.236499,3.270728e+07
1,2025 M11,STE,131.365156,-0.454429,32.198945,0.434835,1.310443,164.854950,1.544695,0.234064,4.813154e+07
2,2025 M12,STE,131.651392,-0.563050,31.740297,0.403174,1.282238,164.514051,1.573478,0.234407,4.906867e+07
3,2026 M01,STE,132.250448,-0.484875,31.022217,0.441344,1.278000,164.507133,1.538778,0.234458,4.206601e+07
4,2026 M02,STE,133.111325,-0.495787,29.284617,0.483337,1.380390,163.763883,1.500602,0.233213,5.189566e+07
5,2026 M03,STE,134.393443,-0.426398,27.853713,0.526439,1.767534,164.114731,1.461419,0.228505,1.078950e+08
6,2026 M04,STE,135.073919,-0.356603,27.833102,0.542930,1.964518,165.057868,1.446427,0.226109,9.062796e+07
7,2026 M05,STE,136.244091,-0.453691,27.038916,0.585181,2.094626,165.509124,1.408017,0.224527,7.272996e+07
8,2026 M06,STE,136.526976,-0.547894,27.070776,0.568244,2.211686,165.829787,1.423415,0.223104,8.233838e+07
9,2026 M07,STE,137.337138,-0.508187,26.272513,0.581499,2.406612,166.089575,1.411365,0.220733,5.938374e+07


In [20]:
# =============================================================================
# CELL 10: EXCEL EXPORT (AGGREGATED ONLY -- no individual loan rows)
# =============================================================================
# RAGU = Model Score + GL Impact + Recovery Impact + LTV Impact + APR Impact
# All components additive. Higher RAGU = lower expected loss.
# MS/LTV/APR from SFS dual-path; GL/Recovery from individual V3 scoring.

METRIC_ROWS = [
    ('Model Score',           'model_score'),
    ('Gross Loss Impact',     'gross_loss_impact'),
    ('Recovery Impact',       'recovery_impact'),
    ('LTV Impact',            'ltv_impact'),
    ('APR Impact',            'apr_impact'),
    ('RAGU Score',            'ragu_score'),
    ('Amount Financed',       'amt_financed'),
    ('Weighted LTV',          'ltv'),
    ('Weighted APR',          'apr'),
]

sheet_name = 'STE V3 Data (M)'
sorted_vintages = sorted(vintage_df['vintage'].unique())

if os.path.exists(EXCEL_OUTPUT):
    wb = openpyxl.load_workbook(EXCEL_OUTPUT)
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name, 0)
else:
    wb = openpyxl.Workbook()
    ws = wb.active
    ws.title = sheet_name

current_row = 1

# Header row with vintage labels
ws.cell(row=current_row, column=1, value='STE')
for col_idx, v in enumerate(sorted_vintages, start=2):
    ws.cell(row=current_row, column=col_idx, value=v)
current_row += 1

# Metric rows
lob_data = vintage_df.set_index('vintage')
for label, col_key in METRIC_ROWS:
    ws.cell(row=current_row, column=1, value=label)
    for col_idx, v in enumerate(sorted_vintages, start=2):
        if v in lob_data.index:
            ws.cell(row=current_row, column=col_idx, value=lob_data.loc[v, col_key])
    current_row += 1

wb.save(EXCEL_OUTPUT)
print(f"Sheet '{sheet_name}': STE x {len(sorted_vintages)} periods")
print(f"Saved to {EXCEL_OUTPUT}")
print(f"\nFormula: RAGU = Model Score + GL Impact + Recovery Impact + LTV Impact + APR Impact")
print(f"  MS/LTV/APR: SFS dual-path (ste_ragu_weekly.txt)")
print(f"  GL/Recovery: V3 individual scoring (ste_ragu_ula.txt + ste_ragu_recovery.txt)")

Sheet 'STE V3 Data (M)': STE x 11 periods
Saved to ste_ragu_v3_production.xlsx

Formula: RAGU = Model Score + GL Impact + Recovery Impact + LTV Impact + APR Impact
  MS/LTV/APR: SFS dual-path (ste_ragu_weekly.txt)
  GL/Recovery: V3 individual scoring (ste_ragu_ula.txt + ste_ragu_recovery.txt)
